This notebook is the Pixeltable [Quickstart](https://docs.pixeltable.com/overview/quick-start), using `uv` only. Each command has `uv run` in front.


## Install


In [6]:
!uv add 'pixeltable[serve]'

Resolved 120 packages in 13ms
Audited 114 packages in 23ms


`[serve]` pulls in FastAPI and uvicorn, which `pxt service` needs.


## Write the file


In [7]:
%%writefile app.py
import pixeltable as pxt
import pixeltable.functions as pxtf
from pixeltable.serving import FastAPIRouter

TableModel = pxt.model_base()


class Docs(TableModel, name='docs'):
    title: pxt.String
    body: pxt.String | None
    title_upper = pxtf.string.upper(title)


ingest = FastAPIRouter(name='ingest')
ingest.add_insert_route(
    Docs,
    path='/docs',
    inputs=[Docs.title, Docs.body],
    outputs=[Docs.title, Docs.title_upper],
)

Overwriting app.py


## Apply, then serve

`pxt init` writes `pixeltable.toml`, which makes this directory the project root. Schema and service refuse a file with no project root.

`pxt schema update` creates the directory and tables. It does not start HTTP. `pxt service update` does not create tables. Apply first.


In [8]:
!uv run pxt init

project root: /Users/alison-pxt/Documents/Github/pxt-champs
already configured by pyproject.toml


In [9]:
!uv run pxt schema update app.py my_app

pxt: 500 ImportError: the psycopg package should be imported before psycopg_binary

--- daemon traceback ---
Traceback (most recent call last):
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/http_server.py", line 126, in _dispatch
    result = handler(req)
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/routes.py", line 508, in schema_diff
    return _SCHEMA_PLAN.validate_python(bridge.schema_diff(body.schema_file, req.resolve_path(body.catalog_dir)))
                                        ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/bridge.py", line 726, in schema_diff
    return _schema_plan(model_bases, schema_file, catalog_dir)
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixe

In [10]:
!uv run pxt service update app.py my_app -f

pxt: 500 ImportError: the psycopg package should be imported before psycopg_binary

--- daemon traceback ---
Traceback (most recent call last):
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/http_server.py", line 126, in _dispatch
    result = handler(req)
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/routes.py", line 530, in service_diff
    bridge.service_diff(body.app_file, req.resolve_path(body.target), otel=body.otel)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alison-pxt/Documents/Github/pxt-champs/.venv/lib/python3.14/site-packages/pixeltable_cli/server/bridge.py", line 560, in service_diff
    diffs = [_service_diff(name, service, app_file, target, otel) for name, service in sorted(services.items())]
             ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alison-pxt/Docu

## Insert a row

`title_upper` is computed on the way in. `pxt service list` prints the URL (the port is assigned):


In [ ]:
!uv run pxt service list


In [ ]:
!curl -X POST http://127.0.0.1:<port>/docs \
  -H 'Content-Type: application/json' \
  -d '{"title": "Hello", "body": "world"}'
